## Model Creation and Evaluation

In [2]:
# import packages 
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('../data/cleaned_data.csv') # upload dataset 

In [18]:
df['start_time'] = pd.to_datetime(df['start_time']) # convert to datetime object 

# extract features 
df['month'] = df['start_time'].dt.month
df['dayofweek'] = df['start_time'].dt.dayofweek
df['day'] = df['start_time'].dt.day

# feature matrix 
X = pd.get_dummies(df[['state', 'Event Type', 'county', 'duration',
                        'month', 'dayofweek', 'day']], drop_first=True)
y = df['hour']

In [19]:
# split into train and test set 
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_unscaled)
X_test_scaled = scaler.transform(X_test_unscaled)

# convert back to dataframe 
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_unscaled.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_unscaled.index)

# cross validation 
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
# define model 
model = LinearRegression()

# cross validation
cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()

# train model 
model.fit(X_train_scaled, y_train)

# predict
y_pred_lr = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred_lr)
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Linear Regression Results
CV R2: 0.15716462094502914
Test R2: 0.16679505656875426
MAE (Hours): 5.456059416775833
RMSE (Hours): 6.65820483225602


These results indicate that the model explains ~17% of the outage timing and is off by roughly 5.5 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

In [7]:
# use unscaled data 
X_train = X_train_unscaled
X_test = X_test_unscaled

# define model 
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# cross validation
cv_r2 = cross_val_score(model, X_train, y_train, cv=10, scoring='r2').mean()

# train model 
model.fit(X_train, y_train)

# predict 
y_pred_rf = model.predict(X_test)

r2 = r2_score(y_test, y_pred_rf)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Random Forest Results
CV R2: 0.7798647484433937
Test R2: 0.7907517439545834
MAE (Hours): 1.5083829564213596
RMSE (Hours): 3.3366615453783606


These results indicate that the model explains ~79% of the outage timing and is off by roughly 1.5 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

In [8]:
# define model
model_xgb = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# cross validation
cv_r2 = cross_val_score(model_xgb, X_train_unscaled, y_train, cv=5, scoring='r2').mean()

# train model 
model_xgb.fit(X_train_unscaled, y_train)

# predict
y_pred_xgb = model_xgb.predict(X_test_unscaled)

r2 = r2_score(y_test, y_pred_xgb)
mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print("XGBoost Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

XGBoost Results
CV R2: 0.5208075523376465
Test R2: 0.5352447032928467
MAE (Hours): 3.830061197280884
RMSE (Hours): 4.972713017645335


These results indicate that the model explains ~54% of the outage timing and is off by roughly 4 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

## Hyperparameter Tuning

In [26]:
# define hyperparameters 
lr_param_dist = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# initialize model 
lr = LinearRegression()


lr_rand = RandomizedSearchCV(
    lr,
    lr_param_dist,
    n_iter=4,   # fewer iterations because model would not run with higher number of iterations    
    cv=5,       # could not increase number of folds without failing to run    
    scoring='r2', # select best model based on R^2 score 
    n_jobs=-1,  # utilize all CPUs when running 
    random_state=42,
)

# train model 
lr_rand.fit(X_train_scaled, y_train)

# predict by best model 
y_pred_lr_tuned = lr_rand.best_estimator_.predict(X_test_scaled)

print("Test R2:", r2_score(y_test, y_pred_lr_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_lr_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr_tuned)))

Test R2: 0.16679505656875426
MAE: 5.456059416775833
RMSE: 6.65820483225602


These results indicate that the model explains ~17% of the outage timing and is off by roughly 5.5 hours. We can see that even after hyperparamter tuning, the results did not change. So we can conclude that the logistic regression model is not an effective model for forecasting time of day of an outage. 

In [29]:
# define hyperparameters 
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# initialize model
rf = RandomForestRegressor(random_state=42)

rf_random = RandomizedSearchCV(
    rf,
    rf_param_grid,
    n_iter=4,      # fewer iterations because model would not run with higher number of iterations    
    cv=3,          # could not increase number of folds without failing to run
    scoring='r2',  # select best model based on R^2 score 
    n_jobs=-1,      # utilize all CPUs 
    random_state=42
)

# train model
rf_random.fit(X_train_scaled, y_train)

# predict by best model 
y_pred_rf_tuned = rf_random.best_estimator_.predict(X_test_scaled)

# Metrics
print("Test R2:", r2_score(y_test, y_pred_rf_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)))

Test R2: 0.7908493335991307
MAE: 1.5082469046658653
RMSE: 3.3358833749996517


These results indicate that the model explains ~79% of the outage timing and is off by roughly 1.5 hours. This fine tuning did not impact the results at all. 

In [22]:
# define hyperparameters 
xgb_param_grid = {
    'n_estimators': [200, 400],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 1.0]
}

# initialize model 
xgb = XGBRegressor(random_state=42, n_jobs=-1)

xgb_random = RandomizedSearchCV(
    xgb,
    xgb_param_grid,
    n_iter=4,     # fewer iterations because model would not run with higher number of iterations  
    cv=3,         # could not increase number of folds without failing to run
    scoring='r2',   # select best model based on R^2 score
    n_jobs=-1,    # utilize all CPUs 
    random_state=42
)

# train model  
xgb_random.fit(X_train_unscaled, y_train)

# predict by best model 
y_pred_xgb_tuned = xgb_random.best_estimator_.predict(X_test_unscaled)

r2 = r2_score(y_test, y_pred_xgb)
mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print("XGBoost Results")
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

XGBoost Results
Test R2: 0.5352447032928467
MAE (Hours): 3.830061197280884
RMSE (Hours): 4.972713017645335


These results indicate that the model explains ~54% of the outage timing and is off by roughly 4 hours. The fine tuning did not impact the results at all. 

## Model Evaluation

In [24]:
# define variables 
y_pred_lr_before = y_pred_lr
y_pred_rf_before = y_pred_rf
y_pred_xgb_before = y_pred_xgb

y_pred_lr_after = y_pred_lr_tuned
y_pred_rf_after = y_pred_rf_tuned
y_pred_xgb_after = y_pred_xgb_tuned  

# define models 
models = ["Linear Regression", "Random Forest", "XGBoost"]
rows = []

# create metric table 
for model, y_before, y_after in zip(
    models,
    [y_pred_lr_before, y_pred_rf_before, y_pred_xgb_before],
    [y_pred_lr_after, y_pred_rf_after, y_pred_xgb_after]
):
    rows.append({
        "Model": model,
        "R2 Before": r2_score(y_test, y_before),
        "R2 After": r2_score(y_test, y_after),
        "MAE Before": mean_absolute_error(y_test, y_before),
        "MAE After": mean_absolute_error(y_test, y_after),
        "RMSE Before": np.sqrt(mean_squared_error(y_test, y_before)),
        "RMSE After": np.sqrt(mean_squared_error(y_test, y_after))
    })

performance_table = pd.DataFrame(rows)
print(performance_table)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 2))
ax.axis('tight')
ax.axis('off')

table = ax.table(cellText=performance_table.values,
                 colLabels=performance_table.columns,
                 cellLoc='center',
                 loc='center')

plt.savefig("../outputs/performance_table.png", dpi=150)
plt.close()

               Model  R2 Before  R2 After  MAE Before  MAE After  RMSE Before  \
0  Linear Regression   0.166795  0.166795    5.456059   5.456059     6.658205   
1      Random Forest   0.790752  0.790984    1.508383   1.507656     3.336662   
2            XGBoost   0.535245  0.558044    3.830061   3.737221     4.972713   

   RMSE After  
0    6.658205  
1    3.334807  
2    4.849206  
